In [1]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/jester_dataset")

In [6]:
df = pd.read_csv("transf_data.csv",
                   header=None)

rating = pd.melt(df, id_vars=0)
rating.columns=["uid", "iid", "rating"]
rating = rating[(rating['rating']<=10) & (rating['rating']>=-10)]



In [7]:
rating

,uid,iid,rating
46200,112,7,-4.45
46233,75,7,-10.00
46282,73,7,-5.76
46286,67,7,9.04
46304,121,7,0.00
...,...,...,...
1216420,33,158,0.00
1216425,67,158,2.73
1216433,50,158,0.41
1216439,26,158,0.65


In [8]:
lowest_rating = rating['rating'].min()
highest_rating = rating['rating'].max()
lowest_rating,highest_rating

(-10.0, 10.0)

In [9]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(df=rating,reader=reader)

In [10]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset()) 

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [11]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
# param_grid ={'k':[20,30,50,70,90],'user_based':[True,False]}

param_grid ={'n_epochs' : np.arange(5, 50, 10),
            'lr_all': np.linspace(0.001, 1, 5),
            'reg_all': np.linspace(0.01, 0.8, 5)}


kfold= KFold(n_splits=5,shuffle=True,random_state=26)
# gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kfold)
# gs=GridSearchCV(surprise.KNNWithZScore,param_grid,measures=['rmse','mae'],cv=kfold)
gs=GridSearchCV(surprise.SVD,param_grid,measures=['rmse','mae'], joblib_verbose = 3,cv=kfold,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:   53.0s
[Parallel(n_jobs=-1)]: Done 256 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 480 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 625 out of 625 | elapsed:  5.5min finished


In [13]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

4.785128615841522
{'n_epochs': 25, 'lr_all': 0.001, 'reg_all': 0.20750000000000002}


In [9]:
user = 50
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
print('List of items rated by user:',u_iid)
print('No. of items rated by user {0} : {1} '.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("Items not rated by the user or those items for which the expected ratings are to be predicted",iids_to_predict)

List of items rated by user: [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50 : 24 
Items not rated by the user or those items for which the expected ratings are to be predicted [   1    2    3 ... 1680 1681 1682]


In [10]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df.sort_values('est_rating',ascending=False)

,iid,est_rating
1475,1500,5.0
1442,1467,5.0
1164,1189,5.0
1097,1122,5.0
1628,1653,5.0
...,...,...
1552,1577,1.0
1551,1576,1.0
1550,1575,1.0
1549,1574,1.0


In [11]:
movies = pd.read_csv('movies_list.csv',encoding='latin-1')

In [17]:
user = 50
u_iid = rating[rating['uid']==user]['iid'].unique()
iids = rating['iid'].unique()
iids_to_predict = np.setdiff1d(iids,u_iid)
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df.sort_values(by = 'est_rating', ascending = False).head(10)

ValueError: 3 columns passed, passed data had 2 columns

In [13]:
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display,clear_output

In [14]:
def predict_user_review(user):
    u_iid = rating[rating['uid']==user]['Iid'].unique()
    iids_to_predict = np.setdiff1d(iids,u_iid)
    testset = [[user,iid,0.] for iid in iids_to_predict]
    predictions = algo.test(testset)
    exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
    df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
    df = df.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on = 'iid', right_on = 'movie id ')
    return df.sort_values(by = 'est_rating', ascending = False).head(10)[['movie id ', ' movie title ', ' release date ']]

In [15]:
rating['uid'].min(),rating['uid'].max()
rating['Iid'].min(),rating['Iid'].max()


(1, 1682)

In [16]:
unique_users = rating['uid'].unique()

In [19]:
output = interact(predict_user_review, user = iids)
display(output)

,movie id,movie title,release date
1578,1599,Someone Else's America (1995),10-May-96
1105,1122,They Made Me a Criminal (1939),01-Jan-39
1274,1293,Star Kid (1997),16-Jan-98
1632,1653,Entertaining Angels: The Dorothy Day Story (1996),27-Sep-96
1446,1467,"Saint of Fort Washington, The (1993)",01-Jan-93
1635,1656,Little City (1998),20-Feb-98
1515,1536,Aiqing wansui (1994),22-Jul-96
1182,1201,Marlene Dietrich: Shadow and Light (1996),02-Apr-96
799,814,"Great Day in Harlem, A (1994)",01-Jan-94
1479,1500,Santa with Muscles (1996),08-Nov-96


interactive(children=(Dropdown(description='user', options=(242, 302, 377, 51, 346, 474, 265, 465, 451, 86, 25…

<function __main__.predict_user_review(user)>